# Coverage-Guided Fuzz Test Input Generation (SuT: `classify_triangle`)

In this exercise, we'll implement a simple **coverage-guided fuzzing** algorithm that generates random test inputs and selects those that increase structural coverages.

In [115]:
import os, sys

# exercises → unit folder → modules → project_root
MODULES_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CODE_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "code")
MODEL_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "model")
DATA_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "data")

if CODE_DIR not in sys.path:
    sys.path.append(CODE_DIR)

if MODEL_DIR not in sys.path:
    sys.path.append(MODEL_DIR)

if DATA_DIR not in sys.path:
    sys.path.append(DATA_DIR)

print("Added to sys.path:", CODE_DIR)
print("Added to sys.path:", MODEL_DIR)
print("Added to sys.path:", DATA_DIR)

from triangle_instrumentation import *

stmt_tracker = StatementCoverageTracker()
branch_tracker = BranchCoverageTracker()
cond_tracker = ConditionCoverageTracker()
path_tracker = PathCoverageTracker()
mcdc_tracker = MCDCCoverageTracker()

print("Five structural coverage trackers have been initialized.")


Added to sys.path: /workspace/modules/exercise_artifacts/code
Added to sys.path: /workspace/modules/exercise_artifacts/model
Added to sys.path: /workspace/modules/exercise_artifacts/data
Five structural coverage trackers have been initialized.


## Random Input Generator

In [129]:
# Pure random generator (baseline)
import random

def generate_random_test_input():
    a = round(random.uniform(0, 10), 2)
    b = round(random.uniform(0, 10), 2)
    c = round(random.uniform(0, 10), 2)
    # a = random.randint(0, 10)
    # b = random.randint(0, 10)
    # c = random.randint(0, 10)
    return a, b, c

## Statement Coverage-Guided Generation

In [130]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    stmt_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        stmt_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    stmt_tracker.run_test(*random_test_input)
    new_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
stmt_tracker.reset()
for test_input in test_inputs:
    stmt_tracker.run_test(*test_input)
stmt_tracker.print_report()

STATEMENT COVERAGE REPORT

Overall Coverage: 54.55% (6/11 items)
Total Tests: 2

| Test input               | (3.87, 2.72, 4.89)   | (0.22, 5.63, 1.04)   | Covered   |
|--------------------------|----------------------|----------------------|-----------|
| sides = sorted([a,b,c])  | O                    | O                    | O         |
| x, y, z = sides          | O                    | O                    | O         |
| return 'invalid' #1      | X                    | X                    | X         |
| return 'invalid' #2      | X                    | O                    | O         |
| return 'equilateral'     | X                    | X                    | X         |
| is_isosceles = ...       | O                    | X                    | O         |
| is_right = ...           | O                    | X                    | O         |
| return 'right_isosceles' | X                    | X                    | X         |
| return 'isosceles'       | X                   

## Branch Coverage-Guided Generation

In [131]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    branch_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        branch_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    branch_tracker.run_test(*random_test_input)
    new_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
branch_tracker.reset()
for test_input in test_inputs:
    branch_tracker.run_test(*test_input)
branch_tracker.print_report()

BRANCH COVERAGE REPORT

Overall Coverage: 58.33% (7/12 items)
Total Tests: 2

| Test input                       | (6.4, 9.62, 8.93)   | (1.07, 8.32, 3.25)   | Covered   |
|----------------------------------|---------------------|----------------------|-----------|
| x <= 0: True                     | X                   | X                    | X         |
| x <= 0: False                    | O                   | O                    | O         |
| x + y <= z: True                 | X                   | O                    | O         |
| x + y <= z: False                | O                   | X                    | O         |
| x == y == z: True                | X                   | X                    | X         |
| x == y == z: False               | O                   | X                    | O         |
| is_isosceles and is_right: True  | X                   | X                    | X         |
| is_isosceles and is_right: False | O                   | X                

## Condition Coverage-Guided Generation

In [132]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    cond_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        cond_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = cond_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    cond_tracker.run_test(*random_test_input)
    new_coverage, _, _ = cond_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
cond_tracker.reset()
for test_input in test_inputs:
    cond_tracker.run_test(*test_input)
cond_tracker.print_report()

CONDITION COVERAGE REPORT

Overall Coverage: 58.33% (7/12 items)
Total Tests: 2

| Test input              | (9.99, 2.23, 5.12)   | (7.97, 8.53, 8.07)   | Covered   |
|-------------------------|----------------------|----------------------|-----------|
| x <= 0: True            | X                    | X                    | X         |
| x <= 0: False           | O                    | O                    | O         |
| x + y <= z: True        | O                    | X                    | O         |
| x + y <= z: False       | X                    | O                    | O         |
| x == y == z: True       | X                    | X                    | X         |
| x == y == z: False      | X                    | O                    | O         |
| x == y: True            | X                    | X                    | X         |
| x == y: False           | X                    | O                    | O         |
| y == z: True            | X                    | X       

## Path Coverage-Guided Generation

In [133]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    path_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        path_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    path_tracker.run_test(*random_test_input)
    new_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
path_tracker.reset()
for test_input in test_inputs:
    path_tracker.run_test(*test_input)
path_tracker.print_report()

PATH COVERAGE REPORT

Overall Coverage: 28.57% (2/7 items)
Total Tests: 2

| Test input                     | (1.06, 4.0, 1.41)   | (4.36, 9.6, 7.91)   | Covered   |
|--------------------------------|---------------------|---------------------|-----------|
| Path: return 'invalid' #1      | X                   | X                   | X         |
| Path: return 'invalid' #2      | O                   | X                   | O         |
| Path: return 'equilateral'     | X                   | X                   | X         |
| Path: return 'right_isosceles' | X                   | X                   | X         |
| Path: return 'isosceles'       | X                   | X                   | X         |
| Path: return 'right'           | X                   | X                   | X         |
| Path: return 'scalene'         | X                   | O                   | O         |
|                                |                     |                     |           |
| Result       

## MC/DC Coverage-Guided Generation

In [134]:
BUDGET = 100

test_inputs = []
base_input = generate_random_test_input()
test_inputs.append(base_input)

for i in range(BUDGET):
    mcdc_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        mcdc_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = mcdc_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    mcdc_tracker.run_test(*random_test_input)
    new_coverage, _, _ = mcdc_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
mcdc_tracker.reset()
for test_input in test_inputs:
    mcdc_tracker.run_test(*test_input)
mcdc_tracker.print_report()

MCDC COVERAGE REPORT

Overall Coverage: 22.22% (2/9 items)
Total Tests: 3

| Test input                 | (3.41, 5.25, 0.82)   | (7.74, 2.2, 6.93)   | (4.68, 3.74, 4.68)   | Covered   |
|----------------------------|----------------------|---------------------|----------------------|-----------|
| Effect of cond1->decision1 | X                    | X                   | X                    | X         |
| Effect of cond2->decision2 | X                    | O                   | O                    | O         |
| Effect of cond3->decision3 | X                    | X                   | X                    | X         |
| Effect of cond4->decision4 | X                    | X                   | X                    | X         |
| Effect of cond5->decision4 | X                    | X                   | X                    | X         |
| Effect of cond6->decision4 | X                    | X                   | X                    | X         |
| Effect of cond4->decision5 | X     

## 🤔 Think: Smarter Random Input Generator

Our `pure` random input generator struggles to achieve high coverage.

**Why?** Random triples `(a, b, c)` frequently violate the triangle inequality (`a + b > c`), producing many "invalid" results and missing important code paths such as `"equilateral"` or `"isosceles"`.

---

### ✅ Your Task

Modify the `generate_random_test_input()` above to **return only integer triples** instead of arbitrary floating-point numbers.

Even this small change introduces an **integer bias**, which significantly increases the probability of hitting meaningful branches.

This is an extremly simple example of a **Biased Random** approach.

You may observe that this biased generation achieves much higher coverage than the pure one.

---

## 🎯 Smarter Random Generators in Fuzzing

The ultimate goal of fuzzing is not just to generate random inputs —  
but to generate **smarter random inputs**.

Below are representative approaches used in practice:

| Approach                         | Required SuT Knowledge                     | Example                          |
|----------------------------------|--------------------------------------------|----------------------------------|
| Pure Random                      | 🟢 None (Black-box)                         | Uniform random sampling          |
| Biased Random                    | 🟢 Very Low (general heuristics only)       | Small-int bias, boundary values  |
| Mutation-Based                   | 🟡 Low (seed inputs required)               | AFL-style bit/byte mutations     |
| Evolutionary / Genetic           | 🟡 Grey-box (fitness such as coverage)      | Coverage-based genetic fuzzing   |
| Grammar-Based                    | 🟠 Input format / grammar knowledge         | JSON/XML grammar fuzzing         |
| Constraint-Based / Symbolic      | 🔴 High (White-box, path constraint access) | KLEE, symbolic execution engines |


---

### 🔎 Reflection

- Why does integer bias improve coverage in this case?
- What kind of bias would help if the input domain were images? Strings? Network packets?
- How much knowledge about the system under test (SuT) should a fuzzing strategy assume?

Keep in mind:

> Pure random is rarely efficient in practice.  
> **The key to effective fuzzing is choosing (or designing) a strategy that fits your SuT.**